# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

In [5]:
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install "vllm==0.8.5" "transformers==4.51.3" "compressed-tensors==0.9.3" bitsandbytes tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 108.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 75.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [1]:
import os
print("Colab:", 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ)
print("Kaggle:", 'KAGGLE_KERNEL_RUN_TYPE' in os.environ)
print("Env vars:", {k: v for k, v in os.environ.items() if 'CUDA' in k.upper() or 'GPU' in k.upper()})

Colab: True
Kaggle: False
Env vars: {'NVIDIA_REQUIRE_CUDA': 'cuda>=12.8 brand=unknown,driver>=470,driver<471 brand=grid,driver>=470,driver<471 brand=tesla,driver>=470,driver<471 brand=nvidia,driver>=470,driver<471 brand=quadro,driver>=470,driver<471 brand=quadrortx,driver>=470,driver<471 brand=nvidiartx,driver>=470,driver<471 brand=vapps,driver>=470,driver<471 brand=vpc,driver>=470,driver<471 brand=vcs,driver>=470,driver<471 brand=vws,driver>=470,driver<471 brand=cloudgaming,driver>=470,driver<471 brand=unknown,driver>=535,driver<536 brand=grid,driver>=535,driver<536 brand=tesla,driver>=535,driver<536 brand=nvidia,driver>=535,driver<536 brand=quadro,driver>=535,driver<536 brand=quadrortx,driver>=535,driver<536 brand=nvidiartx,driver>=535,driver<536 brand=vapps,driver>=535,driver<536 brand=vpc,driver>=535,driver<536 brand=vcs,driver>=535,driver<536 brand=vws,driver>=535,driver<536 brand=cloudgaming,driver>=535,driver<536 brand=unknown,driver>=550,driver<551 brand=grid,driver>=550,driver

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: .venv
 + pip==26.1
Activate with: source .venv/bin/activate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 35.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 18.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 80.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 37.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 98.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 75.3 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 122.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 61.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys
sys.path.insert(0, '/content/.venv/lib/python3.12/site-packages')

In [4]:
import torch
print(torch.__version__)
from vllm import LLM, SamplingParams
print("OK")

2.10.0+cu128


ModuleNotFoundError: No module named 'vllm'

### Run the cell below every time to activate the installed environment.

In [4]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
!pip install -q "vllm[cuda]" --extra-index-url https://download.pytorch.org/whl/cu124

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/1

In [ ]:
import torch
from vllm import LLM, SamplingParams
print(torch.__version__)
print("OK")

INFO 05-03 05:57:28 [__init__.py:239] Automatically detected platform cuda.
2.6.0+cu124
OK


In [ ]:
!pip install -q compressed-tensors==0.9.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.8.5 requires gguf>=0.13.0, but you have gguf 0.10.0 which is incompatible.
vllm 0.8.5 requires llguidance<0.8.0,>=0.7.9; platform_machine == "x86_64" or platform_machine == "arm64" or platform_machine == "aarch64", but you have llguidance 1.3.0 which is incompatible.
vllm 0.8.5 requires numba==0.61.2; python_version > "3.9", but you have numba 0.65.0 which is incompatible.
vllm 0.8.5 requires opentelemetry-api<1.27.0,>=1.26.0, but you have opentelemetry-api 1.41.1 which is incompatible.
vllm 0.8.5 requires opentelemetry-exporter-otlp<1.27.0,>=1.26.0, but you have opentelemetry-exporter-otlp 1.41.1 which is incompatible.
vllm 0.8.5 requires opentelemetry-sdk<1.27.0,>=1.26.0, but you have opentelemetry-sdk 1.41.1 which is incompatib

In [6]:
llm = LLM(
    model=MODEL_ID,
    enable_prefix_caching=True,
    gpu_memory_utilization=0.90,
    max_model_len=20480,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=8192,
    dtype="bfloat16",
)

INFO 05-03 06:38:51 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 20480, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'max_num_batched_tokens': 8192, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


/content/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ERROR 05-03 06:38:52 [registry.py:912] Error in inspecting model architecture 'Qwen3ForCausalLM'
ERROR 05-03 06:38:52 [registry.py:912] Traceback (most recent call last):
ERROR 05-03 06:38:52 [registry.py:912]   File "/content/.venv/lib/python3.12/site-packages/vllm/model_executor/models/registry.py", line 1336, in _run_in_subprocess
ERROR 05-03 06:38:52 [registry.py:912]     returned.check_returncode()
ERROR 05-03 06:38:52 [registry.py:912]   File "/usr/lib/python3.12/subprocess.py", line 502, in check_returncode
ERROR 05-03 06:38:52 [registry.py:912]     raise CalledProcessError(self.returncode, self.args, self.stdout,
ERROR 05-03 06:38:52 [registry.py:912] subprocess.CalledProcessError: Command '['/usr/bin/python3', '-m', 'vllm.model_executor.models.registry']' returned non-zero exit status 1.
ERROR 05-03 06:38:52 [registry.py:912] 
ERROR 05-03 06:38:52 [registry.py:912] The above exception was the direct cause of the following exception:
ERROR 05-03 06:38:52 [registry.py:912] 
ERRO

ValidationError: 1 validation error for ModelConfig
  Value error, Model architectures ['Qwen3ForCausalLM'] failed to be inspected. Please check the logs for more details. [type=value_error, input_value=ArgsKwargs((), {'model': ...nderer_num_workers': 1}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [ ]:
!pip install -q vllm==0.8.5 --no-deps
!pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 114.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 147.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.4/768.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -q "protobuf>=5.28.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 27.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.26.0 requires protobuf<5.0,>=3.19, but you have protobuf 7.34.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
opentelemetry-exporter-gcp-trace 1.11.0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.26.0 which is incompatible.
opentelemetry-exporter-gcp-trace 1.11.0 requires opentelemetry-sdk~=1.30, but you have opentelemetry-sdk 1.26.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-d

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [1]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "/content/drive/MyDrive/CSE-151B/151B_SP26_Competition/data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer


from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-03 19:14:14 [__init__.py:239] Automatically detected platform cuda.


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [2]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [3]:
from typing import Optional

# ── Few-shot examples (compact, format-focused) ────────────────────────────
FEW_SHOT_MATH = (
    "\n\nEXAMPLES OF CORRECT ANSWER FORMAT:\n"
    "Q: If $f(x) = 3x^2 - 2x + 1$, find $f(2)$ = [ANS] and $f(-1)$ = [ANS]\n"
    "A: f(2) = 3(4)-2(2)+1 = 9. f(-1) = 3(1)+2+1 = 6. \\boxed{9, 6}\n"
    "\n"
    "Q: Solve $x^2 - 5x + 6 = 0$. [ANS]\n"
    "A: Factor: (x-2)(x-3)=0. \\boxed{2, 3}\n"
    "\n"
    "Q: Evaluate $\\int_0^1 x^2\\,dx$. [ANS]\n"
    "A: $\\frac{x^3}{3}\\Big|_0^1 = \\frac{1}{3}$. \\boxed{\\frac{1}{3}}\n"
    "\n"
    "Q: Find the population variance of 83, 74, 71, 84, 89, 65. [ANS]\n"
    "A: Mean=77.67, sum of squared deviations=415.33, divide by 6. \\boxed{69.22}\n"
)

FEW_SHOT_MCQ = (
    "\n\nEXAMPLES OF CORRECT ANSWER FORMAT:\n"
    "Q: Which of the following equals $\\sin^2x + \\cos^2x$?\n"
    "A. 0  B. 2  C. 1  D. -1\n"
    "A: By the Pythagorean identity, the answer is 1. \\boxed{C}\n"
    "\n"
    "Q: Which values satisfy $x^2 < 4$? Select all that apply.\n"
    "A. x=3  B. x=1  C. x=-1  D. x=2\n"
    "A: Need |x|<2, so x=1 and x=-1 qualify. \\boxed{BC}\n"
)

# ── System prompts ──────────────────────────────────────────────────────────
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician and problem solver. "
    "Solve problems with rigorous, clear step-by-step reasoning.\n"
    "\n"
    "INSTRUCTIONS:\n"
    "1. Read the problem carefully and identify what is being asked.\n"
    "2. Count the number of [ANS] placeholders — you must provide exactly that many answers.\n"
    "3. Show all work clearly, one step at a time.\n"
    "4. Double-check your arithmetic and algebra before writing the final answer.\n"
    "5. Verify your answer by substituting back into the original problem if possible.\n"
    "\n"
    "ANSWER FORMAT (strictly follow this):\n"
    "- Place your final answer inside \\boxed{}.\n"
    "- Exact form preferred: e.g., \\boxed{\\frac{5}{8}}, \\boxed{2\\sqrt{3}}, \\boxed{\\pi/4}.\n"
    "  Use decimals only if the problem explicitly requires them.\n"
    "- For multiple [ANS] placeholders: put ALL answers comma-separated in ONE box,\n"
    "  in the same order as the placeholders: \\boxed{answer1, answer2, answer3}\n"
    "- For sets: \\boxed{\\{1, 2, 3\\}}\n"
    "- For intervals: \\boxed{[0, 1)} or \\boxed{(-\\infty, 2]}\n"
    "- For True/False: \\boxed{True} or \\boxed{False}\n"
    "- The very last \\boxed{} in your response is what will be graded.\n"
    "- Do NOT write alternative answers or say 'or'. Commit to one final answer.\n"
    + FEW_SHOT_MATH
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and all answer choices carefully.\n"
    "\n"
    "INSTRUCTIONS:\n"
    "1. Work through the problem step by step.\n"
    "2. Eliminate clearly wrong options first.\n"
    "3. Verify your chosen answer satisfies all conditions in the problem.\n"
    "4. Double-check before committing.\n"
    "\n"
    "ANSWER FORMAT (strictly follow this):\n"
    "- Single answer: put ONLY the letter inside \\boxed{}, e.g., \\boxed{C}.\n"
    "- Multiple correct answers: put ALL letters in alphabetical order inside ONE \\boxed{},\n"
    "  e.g., \\boxed{ABD}.\n"
    "- Do NOT include option text, parentheses, or punctuation inside the box.\n"
    "- The very last \\boxed{} in your response is what will be graded.\n"
    + FEW_SHOT_MCQ
)

# ── Domain hint injection ───────────────────────────────────────────────────
def get_domain_hint(question: str) -> str:
    q = question.lower()
    if any(k in q for k in ["integral", "integrate", "antiderivat"]):
        return "\n[Hint: Set up the integral carefully. Check bounds and use correct integration techniques.]"
    if any(k in q for k in ["derivative", "differenti", "rate of change", "tangent line"]):
        return "\n[Hint: Apply differentiation rules carefully. Check chain rule, product rule if needed.]"
    if any(k in q for k in ["limit", "lim "]):
        return "\n[Hint: Check if direct substitution works. If not, try factoring, L'Hôpital, or squeeze theorem.]"
    if any(k in q for k in ["matrix", "eigenvalue", "determinant", "eigenvector", "vector space", "linear transform"]):
        return "\n[Hint: Work through the matrix operations step by step. Check your row operations carefully.]"
    if any(k in q for k in ["probability", "expected value", "distribution", "random variable", "bayes"]):
        return "\n[Hint: Define the sample space and events clearly. Use the appropriate probability formula.]"
    if any(k in q for k in ["modulo", "congruent", "prime", "divisib", "gcd", "lcm"]):
        return "\n[Hint: Apply number theory carefully. Consider modular arithmetic or divisibility rules.]"
    if any(k in q for k in ["series", "converge", "diverge", "sequence"]):
        return "\n[Hint: Identify the type of series and apply the appropriate convergence test.]"
    if any(k in q for k in ["prove", "show that", "verify", "proof"]):
        return "\n[Hint: Structure your proof clearly. State assumptions, apply definitions, justify each step.]"
    if any(k in q for k in ["complex", "imaginary", "real part", "imaginary part"]):
        return "\n[Hint: Use properties of complex numbers. Convert to a+bi form if needed.]"
    if any(k in q for k in ["variance", "standard deviation", "mean", "median", "mode"]):
        return "\n[Hint: Identify whether this is population or sample statistics. Use the correct formula.]"
    return ""


# ── Placeholder counter ─────────────────────────────────────────────────────
def count_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


# ── Main prompt builder ─────────────────────────────────────────────────────
def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""

    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        multi_hint = any(
            kw in question.lower()
            for kw in ["select all", "all that apply", "which of the following are",
                       "choose all", "multiple correct", "all correct", "all of the"]
        )
        format_note = (
            "\n\nThis question may have MULTIPLE correct answers. "
            "List ALL correct letters in alphabetical order inside a single \\boxed{}, e.g., \\boxed{ACD}."
            if multi_hint else
            "\n\nSelect the SINGLE best answer. Put only the letter inside \\boxed{}."
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}{format_note}"

    # Free-form
    n_ans = count_ans_placeholders(question)
    if n_ans > 1:
        ans_instruction = (
            f"\n\nIMPORTANT: This problem has exactly {n_ans} blanks ([ANS] placeholders). "
            f"You MUST provide exactly {n_ans} answers in the SAME ORDER as the placeholders appear. "
            f"Put all {n_ans} answers comma-separated inside a SINGLE \\boxed{{}}: "
            f"\\boxed{{answer1, answer2, ...}}"
        )
    else:
        ans_instruction = "\n\nPut your single final answer inside \\boxed{}."

    domain_hint = get_domain_hint(question)

    user_prompt = (
        question
        + ans_instruction
        + domain_hint
        + "\n\nLet me solve this step by step:"
    )

    return SYSTEM_PROMPT_MATH, user_prompt


# ── Verify with samples ─────────────────────────────────────────────────────
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt (first 200 chars) ──")
    print(sys_p[:200], "...\n")
    print(f"── {label} user prompt ──")
    print(usr_p[:300], "...\n")

── MCQ system prompt (first 200 chars) ──
You are an expert mathematician. Read the problem and all answer choices carefully.

INSTRUCTIONS:
1. Work through the problem step by step.
2. Eliminate clearly wrong options first.
3. Verify your ch ...

── MCQ user prompt ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by one percent
E. Decreased by ten percent
F. Halved
G. Unable to determine
H. Doubled
I. Decreased by  ...

── Free-form system prompt (first 200 chars) ──
You are an expert mathematician and problem solver. Solve problems with rigorous, clear step-by-step reasoning.

INSTRUCTIONS:
1. Read the problem carefully and identify what is being asked.
2. Count  ...

── Free-form user prompt ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS]

IMPORTANT: This problem has exa

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
import sys
print(sys.executable)

/usr/bin/python3


In [4]:
import time, threading
from tqdm.auto import tqdm

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Spinner so you know it's not frozen
done = threading.Event()
def spinner():
    pbar = tqdm(desc="Loading model", bar_format="{desc}: {elapsed}")
    while not done.is_set():
        pbar.update(0)
        time.sleep(1)
    pbar.close()

t = threading.Thread(target=spinner, daemon=True)
t.start()
t0 = time.time()

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.90,
    max_model_len=20480,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=8192,
    dtype="bfloat16",
)

done.set()
print(f"Model loaded in {time.time()-t0:.1f}s")

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS, temperature=0.6, top_p=0.95, top_k=20,
    min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0,
)
print("Model loaded.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

INFO 05-03 19:18:37 [weight_utils.py:281] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 17.583021 seconds


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-03 19:18:41 [gpu_model_runner.py:1347] Model loading took 2.5431 GiB and 23.795415 seconds
INFO 05-03 19:18:55 [backends.py:420] Using cache directory: /root/.cache/vllm/torch_compile_cache/0764232330/rank_0_0 for vLLM's torch.compile
INFO 05-03 19:18:55 [backends.py:430] Dynamo bytecode transform time: 14.21 s
INFO 05-03 19:19:02 [backends.py:136] Cache the graph of shape None for later use
INFO 05-03 19:19:47 [backends.py:148] Compiling a graph for general shape takes 49.87 s
INFO 05-03 19:20:28 [monitor.py:33] torch.compile takes 64.09 s in total
INFO 05-03 19:20:29 [kv_cache_utils.py:634] GPU KV cache size: 489,904 tokens
INFO 05-03 19:20:29 [kv_cache_utils.py:637] Maximum concurrency for 20,480 tokens per request: 23.92x
INFO 05-03 19:21:33 [gpu_model_runner.py:1686] Graph capturing finished in 64 secs, took 1.13 GiB
INFO 05-03 19:21:33 [core.py:159] init engine (profile, create kv cache, warmup model) took 172.85 seconds
INFO 05-03 19:21:33 [core_client.py:439] Core engin

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
!pip install -U bitsandbytes>=0.46.1 transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 943 questions...


Processed prompts:   0%|          | 0/943 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Max prompt length: {inputs['input_ids'].shape[1]}")

Input shape: torch.Size([5, 319])
Max prompt length: 319


In [ ]:
import torch
print(torch.cuda.is_available())  # must be True before doing anything else

True


### Generate with Transformers (for Datahub)

In [ ]:
from tqdm.auto import tqdm
from transformers import TextStreamer
import time

# Fix padding for decoder-only batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions (batched)...")
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=16384,
).to(llm.device)
print(f"Input shape: {inputs['input_ids'].shape}")  # sanity check

pbar = tqdm(total=MAX_TOKENS, desc="Tokens generated")

class TqdmStreamer(TextStreamer):
    def put(self, value):
        if value.ndim > 1:
            return
        pbar.update(1)
    def end(self):
        pass

streamer = TqdmStreamer(tokenizer, skip_prompt=True)

t0 = time.time()
with torch.no_grad():
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        do_sample=True,
        streamer=streamer,
    )
pbar.close()
elapsed = time.time() - t0
print(f"Total generation time: {elapsed:.1f}s ({elapsed/len(prompts):.1f}s per prompt avg)")

responses = []
for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions (batched)...
Input shape: torch.Size([5, 410])


Tokens generated:   0%|          | 0/32768 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id": [item["id"] for item in data[:len(responses)]],
    "response": responses,
})
df.to_csv("submission.csv", index=False)
print(f"Saved {len(df)} rows to submission.csv")

Saved 5 rows to submission.csv


In [ ]:
import pandas as pd

df = pd.read_csv("submission.csv")
responses = df["response"].tolist()

print(f"Loaded {len(responses)} responses")

Loaded 5 responses


In [ ]:
responses

['Okay, let\'s see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let\'s recall what the positive even whole numbers are. The first few are 2, 4, 6, ibility, so the nth positive even whole number is 2n. Wait, right, because the first even number is 2 (which is 2×1), the second is 4 (2×2), the third is 6 (2×3), so the kth positive even whole number is 2k. \n\nSo the first 325 positive even whole numbers would be 2, 4, 6, ..., up to the 325th term. Let\'s write that as a sequence: a₁ = 2, a₂ = 4, a₃ = 6, ..., aₙ = 2n. So the nth term is 2n. \n\nWe need the sum of the first 325 terms. So that\'s S = 2 + 4 + 6 + ... + 2×325. Let\'s factor out the 2: S = 2×(1 + 2 + 3 + ... + 325). \n\nAh, right! The sum of the first n positive integers is n(n+1)/2. So here, n is 325. So the sum inside the parentheses is 325×326/2. Then multiply by 2. Let\'s compute that.\n\nFirst, let\'s write the formula for the sum of the first n even numbers. Wait, actually, the sum of 

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [10]:
!pip install -q "antlr4-python3-runtime==4.11.1" --force-reinstall

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.


In [ ]:
import importlib
import judger as _judger_mod
importlib.reload(_judger_mod)
from judger import Judger

In [ ]:
import re, sys
import sympy as sp
from sympy.parsing.latex import parse_latex as _orig_parse_latex

# Patch parse_latex BEFORE importing judger
def safe_parse_latex(s):
    if s.strip() == "\\pi":
        return sp.pi
    return _orig_parse_latex(s)

import sympy.parsing.latex as _latex_mod
_latex_mod.parse_latex = safe_parse_latex

sys.path.insert(0, "/content/drive/MyDrive/CSE-151B/151B_SP26_Competition")

import judger as _judger_mod
_judger_mod.parse_latex = safe_parse_latex

from judger import Judger
judger = Judger(strict_extract=False)


def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
print("=" * 50)
print("PER-QUESTION BREAKDOWN")
print("=" * 50)
for i, r in enumerate(results):
    resp = responses[i]
    has_think_close = "</think>" in resp
    after_think = resp.split("</think>")[-1] if has_think_close else ""
    has_boxed = "\\boxed{" in after_think

    # Extract what the judger would see
    extracted = judger.extract_ans(resp)

    print(f"\n── id={r.get('id', i)} | {'MCQ' if r['is_mcq'] else 'Free'} | {'✓' if r['correct'] else '✗'} ──")
    print(f"  </think> closed:    {has_think_close}")
    print(f"  \\boxed after think: {has_boxed}")
    print(f"  Response length:    {len(resp)} chars")
    print(f"  Gold answer:        {r.get('gold', 'N/A')}")
    print(f"  Extracted by judge: {extracted!r}")
    print(f"  Last 150 chars:     ...{resp[-150:]}")

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path("/content/drive/MyDrive/CSE-151B/151B_SP26_Competition/results/starter_results.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

In [ ]:
import pandas as pd

df = pd.DataFrame([{"id": r["id"], "response": r["response"]} for r in results])
csv_path = "/content/drive/MyDrive/CSE-151B/151B_SP26_Competition/results/submission.csv"
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows to {csv_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!